In [1]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch
from dotenv import load_dotenv
load_dotenv()

C:\Users\pthorat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
import os
password= os.getenv("Elastic_pass")
username= os.getenv("username")
es = Elasticsearch(
    "https://localhost:9200",
    basic_auth=(username, password),
    verify_certs=False
)

# print(es.info())

{'name': '2020AI-050', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'nz0GDvzFQ_i29N0F_Eba1Q', 'version': {'number': '9.4.1', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': '3c7c6027c5769d860d87448e2749f4c550a239da', 'build_date': '2026-05-08T10:08:29.383338563Z', 'build_snapshot': False, 'lucene_version': '10.4.0', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


C:\Users\pthorat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\elasticsearch\_sync\client\__init__.py:326: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(
C:\Users\pthorat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [89]:
df= pd.read_csv("rxdx_aug_2026_indexed_data_med_mod.csv")

In [90]:
model= SentenceTransformer('pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10803.63it/s]


In [91]:
faiss_index= faiss.read_index("rxdx_aug_2026_index_med_mod_updated.faiss")
ES_INDEX_NAME="rxdx_aug2026_index"

In [93]:
def semantic_search(query, top_k=10):
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32").reshape(1, -1)
    D, I = faiss_index.search(query_embedding, top_k)
    # print(I)
    results = []
    for rank, idx in enumerate(I[0]):
        results.append({
            "id": int(idx),
            "rank": rank + 1,
            "description": df.iloc[idx]["description"],
            "HCPCS": df.iloc[idx]["HCPCS"],
            "NDC": df.iloc[idx]["NDC"]
        })
    return results

In [94]:
def syntaticES_search(query, top_k=10):
    response = es.search(
        index=ES_INDEX_NAME,
        body={
            "size": top_k,
            "query": {
                "multi_match": {
                    "query": query,
                    "fields": [
                        "description",
                        "HCPCS^3",
                        "NDC^3"
                    ]
                }
            }
        }
    )

    results = []

    for rank, hit in enumerate(response["hits"]["hits"]):
        results.append({
            "id": int(hit["_id"]),
            "rank": rank + 1,
            "bm25_score": hit["_score"],
            "description": hit["_source"]["description"],
            "HCPCS":hit["_source"]["HCPCS"],
            "NDC": hit["_source"]["NDC"]
        })

    return results

In [95]:
from whoosh import index
from whoosh.qparser import MultifieldParser, OrGroup

ix = index.open_dir(r"C:\temp\whoosh_index")

In [96]:
def whoosh_search(query, top_k=10):

    results_list = []

    with ix.searcher() as searcher:

        parser = MultifieldParser(
            ["description","doc_id", "ndc", "hcpcs", "content"],
            schema=ix.schema,
            group=OrGroup
        )

        parsed_query = parser.parse(query)

        results = searcher.search(parsed_query, limit=top_k)

        for rank, r in enumerate(results):

            results_list.append({
                "id": int(r["doc_id"]),
                
                "rank": rank + 1,

                "description": r["description"],

                "NDC": r["ndc"],

                "HCPCS": r["hcpcs"],

                "bm25_score": r.score
            })

    return results_list

In [ ]:
# def hybrid_search(query, top_k=5):
#     semantic_results = semantic_search(query, top_k=10)
#     # syntatic_result = syntaticES_search(query, top_k=10)
#     syntatic_result = whoosh_search(query, top_k=10)
#     combined_scores = {}
#     k = 60
#     # Process semantic results
#     for result in semantic_results:
#         doc_id = result["id"]
#         rank = result["rank"]
#         rrf_score = 1 / (k + rank)
#         if doc_id not in combined_scores:
#             combined_scores[doc_id] = {
#                 "rrf_score": 0,
#                 "description": result["description"],
#                 "HCPCS":result["HCPCS"],
#                 "NDC": result["NDC"]
#             }
            
#         combined_scores[doc_id]["rrf_score"] += rrf_score

#     # Process BM25 results
#     for result in syntatic_result:
#         doc_id = result["id"]
#         rank = result["rank"]
#         rrf_score = 1 / (k + rank)

#         if doc_id not in combined_scores:
#             combined_scores[doc_id] = {
#                 "rrf_score": 0,
#                 "description": result["description"],
#                 "HCPCS": result["HCPCS"],
#                 "NDC": result["NDC"]
#             }
#         combined_scores[doc_id]["rrf_score"] += rrf_score

#     # Sort by RRF score
#     ranked_results = sorted(
#         combined_scores.items(),
#         key=lambda x: x[1]["rrf_score"],
#         reverse=True
#     )

#     final_results = ranked_results[:top_k]
#     return final_results

In [ ]:
# # query= "63323-0565-93"
# query="Aneshthesia 00024-5915-20"
# # query= "Aneshthesia for head"
# hybrid_search(query)


In [97]:
from sentence_transformers import CrossEncoder
rank_model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2'
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1555.49it/s]


In [114]:
def hybrid_search(query, top_k=5):
    semantic_results = semantic_search(query, top_k=10)
    syntatic_result = whoosh_search(query, top_k=10)
    combined_results = {}

    for result in semantic_results:
        doc_id = result["id"]
        if doc_id not in combined_results:
            combined_results[doc_id] = {
                "id": doc_id,
                "description": result["description"],
                "HCPCS": result["HCPCS"],
                "NDC": result["NDC"]
            }

    for result in syntatic_result:
        doc_id = result["id"]
        if doc_id not in combined_results:
            combined_results[doc_id] = {
                "id": doc_id,
                "description": result["description"],
                "HCPCS": result["HCPCS"],
                "NDC": result["NDC"]
            }

    # Convert dict to list
    candidates = list(combined_results.values())

    # Create text for reranking
    candidate_texts = []

    for item in candidates:
        text = f"""
        Description: {item['description']}
        HCPCS: {item['HCPCS']}
        NDC: {item['NDC']}
        """

        candidate_texts.append(text)

    # Create query-document pairs
    pairs = [
        [query, text]
        for text in candidate_texts
    ]

    scores = rank_model.predict(pairs)

    # Attach scores
    ranked_results = []

    for item, score in zip(candidates, scores):
        item["rerank_score"] = float(score)
        ranked_results.append(item)

    # Sort by reranker score
    ranked_results = sorted(
        ranked_results,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # Return top_k
    return ranked_results[:top_k]

In [115]:
query= "Aneshthesia 00024-5915-20"
whoosh_search(query)

[{'id': 95559,
  'rank': 1,
  'description': 'DUPILUMAB 2 SYRINGE, GLASS IN 1 CARTON / 2 ML IN 1 SYRINGE, GLASS',
  'NDC': '00024-5915-20',
  'HCPCS': '',
  'bm25_score': 26.175454132208195},
 {'id': 95562,
  'rank': 2,
  'description': 'DUPIXENT 300 MG/2ML SOLUTION',
  'NDC': '00024-5915-20',
  'HCPCS': '',
  'bm25_score': 26.175454132208195},
 {'id': 174058,
  'rank': 3,
  'description': 'DUPIXENT 300 MG/2ML INJECTION',
  'NDC': '00024-5915-20',
  'HCPCS': '',
  'bm25_score': 26.175454132208195},
 {'id': 216688,
  'rank': 4,
  'description': 'DUPILUMAB 300 MG/2ML INJECTION',
  'NDC': '00024-5915-20',
  'HCPCS': '',
  'bm25_score': 26.175454132208195},
 {'id': 291893,
  'rank': 5,
  'description': 'DUPILUMAB 300 MG/2ML SOLUTION',
  'NDC': '00024-5915-20',
  'HCPCS': '',
  'bm25_score': 26.175454132208195},
 {'id': 56306,
  'rank': 6,
  'description': 'DUPIXENT 2 SYRINGE, GLASS IN 1 CARTON / 2 ML IN 1 SYRINGE, GLASS',
  'NDC': '00024-5915-20',
  'HCPCS': 'J0717',
  'bm25_score': 23.222

In [120]:
query= "63323-0565-93"
# query="Tray,Guidewire Powerpicc Solo 4fr Single Lumen CatheterWith 70cm 3194355"
# # query= "THYROTROPIN INJ 65219-0368-01"
hybrid_search(query)

[{'id': 17649,
  'description': 'ENOXAPARIN 15 MG INJ',
  'HCPCS': '',
  'NDC': '63323-0565-93',
  'rerank_score': 7.406213760375977},
 {'id': 96201,
  'description': 'ENOXAPARIN INJ 40 MG',
  'HCPCS': '',
  'NDC': '63323-0565-93',
  'rerank_score': 7.34603214263916},
 {'id': 17654,
  'description': 'ENOXAPARIN 40 MG INJ',
  'HCPCS': '',
  'NDC': '63323-0565-93',
  'rerank_score': 7.335321426391602},
 {'id': 96211,
  'description': 'ENOXAPARIN SODIUM INJ 100 MG/ML',
  'HCPCS': '',
  'NDC': '63323-0565-93',
  'rerank_score': 7.000194549560547},
 {'id': 96193,
  'description': 'ENOXAPARIN 300 MG/3 ML INJ',
  'HCPCS': '',
  'NDC': '63323-0565-93',
  'rerank_score': 6.920136451721191}]